# M3-CLV-NV 구성형 가치 그래프 — Dunnhumby validation

M1(이진 그래프)과 M3-CLV-NV(동일 엣지·동일 학습설정, 엣지 가중치만 변경)를 seed 42 validation에서 비교합니다. test·holdout은 생성하지 않습니다.

## 1. 실행 환경 고정

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess

drive.mount('/content/drive')
REVIEWED_SHA = '79a5dd7905ae8917c76107b3b9d810ab00e22a10'
REPO = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if not REPO.exists():
    subprocess.run([
        'git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REVIEWED_SHA], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert head == REVIEWED_SHA, (head, REVIEWED_SHA)
os.chdir(REPO)
print('reviewed source:', head)

## 2. validation 설정 확인

In [ ]:
import json, torch
from lightgcn_clv_m3_nv import (
    configure_m3_clv_nv_dunnhumby_run, preflight_summary, run_experiment,
)

assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
cfg = configure_m3_clv_nv_dunnhumby_run(
    EVAL_TEST=False,
    EVAL_HOLDOUT=False,
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))

## 3. M1과 M3-CLV-NV 한 번에 실행

이 셀을 한 번만 실행하세요. 기존 M1 체크포인트가 있으면 재사용하고, M3만 새로 학습합니다.

In [ ]:
result_df = run_experiment(cfg)

## 4. 결과 확인

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50', 'revenue@10', 'revenue@20', 'revenue@50',
    'arp@10', 'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10', 'value_alignment',
]
validation = result_df[result_df['split'].eq('val')].copy()
display(validation[columns].sort_values(['role', 'model_id']))
print('screening 판정:', result_df.attrs['screening_decision'])
print('결과 폴더:', result_df.attrs['out_dir'])